# Attention Tensor Shapes

Walks a batch through embeddings, causal self-attention (both plain
multi-head and grouped-query variants), and a full Transformer block,
printing the tensor shape after every stage. Useful for building intuition
and as a live sanity check while modifying `src/model/`.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch

from src.model.config import GPTConfig
from src.model.embedding import TokenEmbedding
from src.model.attention import CausalSelfAttention
from src.model.block import TransformerBlock

cfg = GPTConfig(vocab_size=512, block_size=32, d_model=64, n_layers=2,
                 n_heads=4, n_kv_heads=4, d_ff=128, dropout=0.0, attn_dropout=0.0)
print(cfg)

In [ ]:
B, T = 2, 10
tokens = torch.randint(0, cfg.vocab_size, (B, T))

embed = TokenEmbedding(cfg)
x = embed(tokens)
print(f'tokens:          {tuple(tokens.shape)}')
print(f'after embedding: {tuple(x.shape)}   (batch, seq_len, d_model)')

In [ ]:
attn = CausalSelfAttention(cfg)
out, present_kv = attn(x, use_cache=True)
k, v = present_kv
print(f'attention output: {tuple(out.shape)}   (batch, seq_len, d_model)')
print(f'cached keys:       {tuple(k.shape)}   (batch, n_kv_heads, seq_len, head_dim)')
print(f'cached values:     {tuple(v.shape)}')

## Grouped-Query Attention: fewer KV heads than query heads

Notice the cached K/V tensors shrink proportionally to `n_kv_heads`, which
is the entire point of GQA -- a smaller KV cache at inference time -- while
the attention *output* shape is unchanged.

In [ ]:
cfg_gqa = GPTConfig(vocab_size=512, block_size=32, d_model=64, n_layers=2,
                     n_heads=4, n_kv_heads=1, d_ff=128, dropout=0.0, attn_dropout=0.0)
attn_gqa = CausalSelfAttention(cfg_gqa)
x_gqa = TokenEmbedding(cfg_gqa)(tokens)
out_gqa, (k_gqa, v_gqa) = attn_gqa(x_gqa, use_cache=True)
print(f'MQA (n_kv_heads=1) cached keys: {tuple(k_gqa.shape)}  vs MHA: {tuple(k.shape)}')

In [ ]:
block = TransformerBlock(cfg)
out, _ = block(x)
print(f'transformer block output: {tuple(out.shape)}   (unchanged shape, residual stream)')